# GraphRAG Retrievers for Aircraft Maintenance

This notebook demonstrates retrieval strategies for GraphRAG applications, progressing from simple vector search to graph-enhanced retrieval that leverages your aircraft topology.

**Prerequisites:** Complete [01 Data and Embeddings](01_data_and_embeddings.ipynb) first.

**Learning Objectives:**
- Set up a VectorRetriever using Neo4j's vector index
- Perform semantic similarity searches over maintenance procedures
- Use GraphRAG to combine vector search with LLM-generated answers
- Create custom Cypher queries with VectorCypherRetriever for richer context
- Connect maintenance knowledge to your aircraft topology (Aircraft -> System -> Component)

---

## Retrieval Strategies Overview

We'll explore two retrieval approaches:

1. **VectorRetriever** - Simple semantic search using embeddings
   - Finds maintenance procedures by meaning similarity
   - Returns raw text for LLM context

2. **VectorCypherRetriever** - Graph-enhanced semantic search
   - Uses vector search as entry point
   - Traverses graph relationships to aircraft topology
   - Returns structured data (systems, components) alongside text

## Section 1: Configuration

Your Neo4j credentials come from the Databricks secret scope that notebook 01 created, so there is nothing to type here. The cell below derives the same scope name from `current_user()` and reads the URI, username, and password out of it.

Run [01 Data and Embeddings](01_data_and_embeddings.ipynb) first if you have not. The cell raises a clear error when the scope is missing.

Databricks redacts secret values in notebook output, so the URI prints as `[REDACTED]`. That is expected, not a bug. The value is intact in Python and the Neo4j connection works.

In [ ]:
# ==================================================
# CONFIGURATION - Neo4j credentials from notebook 01
# ==================================================

from data_utils import read_neo4j_secrets, secret_scope_name

NEO4J_DATABASE = "neo4j"  # Neo4j database to use (Aura default is "neo4j")

SECRET_SCOPE = secret_scope_name(spark)
print(f"Secret scope: {SECRET_SCOPE}")

credentials = read_neo4j_secrets(dbutils, SECRET_SCOPE)
NEO4J_URI = credentials["uri"]
NEO4J_USERNAME = credentials["username"]
NEO4J_PASSWORD = credentials["password"]

print("Configuration ready!")
print(f"Neo4j URI: {NEO4J_URI}")

## Setup

Import required modules and initialize connections.

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever
from neo4j_graphrag.generation import GraphRAG

from data_utils import Neo4jConnection, get_llm, get_embedder, EMBEDDING_DIMENSIONS, format_operating_limit_record

## Connect to Neo4j

Create and verify the connection to your Neo4j graph database using the credentials from Section 1.

In [ ]:
neo4j = Neo4jConnection(uri=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE).verify()
driver = neo4j.driver

# Show graph statistics
neo4j.get_graph_stats()

## Initialize LLM and Embedder

Set up the Large Language Model (LLM) and embedding model for GraphRAG workflows.

- **LLM**: Uses Databricks Foundation Model APIs (Llama 3.3 70B)
- **Embedder**: Uses Databricks Foundation Model APIs (BGE-large)

In [ ]:
llm = get_llm()
embedder = get_embedder()

print(f"LLM initialized: {llm.model_id}")
print(f"Embedder initialized: {embedder.model_id}")

---

# Part 1: Vector Retriever

The VectorRetriever performs semantic search over your Neo4j knowledge graph. Instead of keyword matching, it finds the most contextually similar maintenance procedures to your query.

## Initialize Vector Retriever

Set up the vector-based retriever for semantic search over maintenance chunks.

In [ ]:
INDEX_NAME = "maintenanceChunkEmbeddings"

vector_retriever = VectorRetriever(
    driver=driver,
    neo4j_database=NEO4J_DATABASE,
    index_name=INDEX_NAME,
    embedder=embedder,
    return_properties=['text']
)

print("VectorRetriever initialized")

The **VectorRetriever** class:
- Connects to Neo4j using the provided `driver`
- Uses the `maintenanceChunkEmbeddings` vector index for semantic retrieval
- The `embedder` generates embeddings for the query
- Returns the `text` property from matching chunks

> **Tip:** You can modify the `return_properties` list to include additional properties like `index` for chunk ordering.

## Simple Vector Search

Test the vector search by retrieving the top 5 most relevant maintenance procedures for a given query.

In [ ]:
query = "What are the steps to troubleshoot engine vibration?"
result = vector_retriever.search(query_text=query, top_k=5)

print(f"Query: \"{query}\"\n")
print(f"Number of results returned: {len(result.items)}\n")
print("=" * 70)

for item in result.items:
    print(f"\nScore: {item.metadata['score']:.4f}")
    print(f"Content: {item.content[0:200]}...")
    print(f"ID: {item.metadata['id']}")

**How it works:**
1. The query is converted to an embedding vector
2. `vector_retriever.search()` finds the top 5 matches based on vector similarity
3. Results show the similarity score, content snippet, and chunk ID

> **Tip:** Inspecting returned results helps verify relevance and adjust your chunking or embedding strategy.

## GraphRAG Pipeline

The `GraphRAG` class combines a Large Language Model (LLM) with a vector-based retriever to answer maintenance questions using both semantic search and generative reasoning.

In [ ]:
query = "What are the normal EGT operating limits for the V2500 engine at different power settings?"

rag = GraphRAG(
    llm=llm,
    retriever=vector_retriever
)

response = rag.search(
    query,
    retriever_config={"top_k": 5},
    return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)

print(f"Query: \"{query}\"\n")
print(f"Number of chunks used: {len(response.retriever_result.items)}\n")
print("=" * 70)
print("\nAnswer:")
print(response.answer)

**How it works:**
1. The retriever (`vector_retriever`) finds the most relevant maintenance chunks
2. The LLM uses the retrieved context to generate a natural language answer
3. The `return_context=True` option lets you see what context was used

The GraphRAG pipeline provides context-aware, accurate answers grounded in your maintenance documentation.

---

**Try different queries:**
- What should I check if there's a fuel starvation warning?
- How often should the hydraulic fluid be sampled?
- What fault codes indicate bearing wear?

---

# Part 2: Vector Cypher Retriever

The VectorCypherRetriever enhances vector search with custom Cypher queries, enabling you to traverse graph relationships and return richer, more contextual answers.

This approach is ideal when:
- Questions involve relationships between maintenance procedures and aircraft components
- You want structured data alongside text context
- Graph traversal can connect documentation to your aircraft topology

## Example 1: Document Context Enrichment

Create a VectorCypherRetriever that returns document metadata alongside the matching chunks, providing context about the source.

In [ ]:
# Custom Cypher query to enrich results with document metadata
document_context_query = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
RETURN 
    doc.documentId AS document_id,
    doc.aircraftType AS aircraft_type,
    doc.title AS document_title,
    node.index AS chunk_index,
    node.text AS context
"""

document_retriever = VectorCypherRetriever(
    driver=driver,
    neo4j_database=NEO4J_DATABASE,
    index_name=INDEX_NAME,
    embedder=embedder,
    retrieval_query=document_context_query
)

print("VectorCypherRetriever initialized with document context query")

**How this query works:**

- Matches text chunks (`node`) to their source document
- Returns: document ID, aircraft type, title, chunk index, and context text

This provides traceability back to the source document for each retrieved chunk.

In [ ]:
query = "What are the hydraulic system pressure limits?"

rag = GraphRAG(llm=llm, retriever=document_retriever)
response = rag.search(
    query,
    retriever_config={"top_k": 3},
    return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)

print(f"Query: \"{query}\"\n")
print(f"Number of results: {len(response.retriever_result.items)}\n")
print("=" * 70)
print("\nAnswer:")
print(response.answer)

In [ ]:
# View the enriched context used by the LLM
print("Context used:")
print("=" * 70)
for item in response.retriever_result.items:
    print(f"\n{item.content}")

Notice how the context includes document metadata (document ID, aircraft type, title) alongside the text. This enables more specific answers with source attribution.

> **Tip:** Modify `top_k` to see how changing the result count affects answer quality.

## Example 2: Adjacent Chunk Retrieval

Leverage the `NEXT_CHUNK` relationships to retrieve surrounding context, providing the LLM with more complete procedure information.

In [ ]:
# Custom Cypher to include previous and next chunks for better context
adjacent_chunks_query = """
WITH node
OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
RETURN 
    doc.documentId AS document_id,
    node.index AS chunk_index,
    COALESCE(prev.text, '') AS previous_context,
    node.text AS main_context,
    COALESCE(next.text, '') AS next_context
"""

adjacent_retriever = VectorCypherRetriever(
    driver=driver,
    neo4j_database=NEO4J_DATABASE,
    index_name=INDEX_NAME,
    embedder=embedder,
    retrieval_query=adjacent_chunks_query
)

print("VectorCypherRetriever initialized with adjacent chunks query")

In [ ]:
query = "How do I perform the engine vibration diagnostic flow?"

rag = GraphRAG(llm=llm, retriever=adjacent_retriever)
response = rag.search(
    query,
    retriever_config={"top_k": 3},
    return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)

print(f"Query: \"{query}\"\n")
print(f"Number of results: {len(response.retriever_result.items)}\n")
print("=" * 70)
print("\nAnswer:")
print(response.answer)

**How this works:**

1. **Semantic Search:** Finds top-k text chunks relevant to the query
2. **Graph Traversal:** For each matched chunk:
   - Follows `NEXT_CHUNK` backward to get previous context
   - Follows `NEXT_CHUNK` forward to get next context
3. **Returns:** Previous, main, and next chunks as combined context

**Why this is powerful:**
- Procedures often span multiple chunks
- Adjacent context provides complete step sequences
- Decision trees and troubleshooting flows are better understood with surrounding content

## Example 3: Connecting to Aircraft Topology

This example demonstrates the full power of GraphRAG: starting from a semantically matched chunk, traversing through the Document's `APPLIES_TO` relationship to reach the Aircraft and its Systems and Components.

In notebook 01, we created `APPLIES_TO` relationships linking Documents to Aircraft. This explicit graph traversal replaces ad-hoc text pattern matching with structured relationship following.

In [ ]:
# Traverse from matched chunk through Document -> Aircraft -> System -> Component
system_context_query = """
WITH node
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)-[:APPLIES_TO]->(a:Aircraft)
MATCH (a)-[:HAS_SYSTEM]->(s:System)
OPTIONAL MATCH (s)-[:HAS_COMPONENT]->(comp:Component)

WITH node, doc, a, s, comp
RETURN 
    doc.documentId AS document_id,
    doc.aircraftType AS aircraft_type,
    a.tail_number AS aircraft,
    COLLECT(DISTINCT s.name)[0..3] AS systems,
    COLLECT(DISTINCT comp.name)[0..3] AS components,
    node.text AS context
"""

system_retriever = VectorCypherRetriever(
    driver=driver,
    neo4j_database=NEO4J_DATABASE,
    index_name=INDEX_NAME,
    embedder=embedder,
    retrieval_query=system_context_query
)

print("VectorCypherRetriever initialized with system context query")

In [ ]:
query = "What maintenance is required for the engine fuel pump?"

rag = GraphRAG(llm=llm, retriever=system_retriever)
response = rag.search(
    query,
    retriever_config={"top_k": 3},
    return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)

print(f"Query: \"{query}\"\n")
print(f"Number of results: {len(response.retriever_result.items)}\n")
print("=" * 70)
print("\nAnswer:")
print(response.answer)

In [ ]:
# View the graph-connected context
print("Context with system connections:")
print("=" * 70)
for item in response.retriever_result.items:
    print(f"\n{item.content}")

**How this works:**

1. **Semantic Search:** Finds maintenance chunks about the query topic
2. **APPLIES_TO Traversal:** Follows Document -> Aircraft relationship
3. **Topology Traversal:** Walks Aircraft -> System -> Component hierarchy
4. **Returns:** Document metadata, aircraft, related systems/components, and context text

This is the GraphRAG advantage: the retriever returns not just the matching text but the structured context surrounding it in the knowledge graph.

## Example 4: Operating Limits as Structured Data

Lab 2 loaded 20 canonical `OperatingLimit` nodes from the manuals' limit tables, and notebook 01 wired each one to the sensors it judges with `HAS_LIMIT`. This example traverses the full chain: Chunk -> Document -> Aircraft -> System -> Sensor -> OperatingLimit.

`OperatingLimit` is the right population here. The question this retriever answers is what the documented threshold for a sensor is, and that answer has to be the hand-transcribed number rather than the language model's reading of it. Notebook 01 linked the extraction output in parallel as `ExtractedLimit` behind `HAS_EXTRACTED_LIMIT`. Swap the relationship type and label in the Cypher below to see what the model read instead, which is a useful comparison and a poor default.

The retriever returns both the relevant text AND the structured operating limit data, so the LLM can give precise answers with specific numeric thresholds.

In [ ]:
# Traverse from chunk through the full graph to the canonical operating limits.
# HAS_LIMIT reaches Lab 2's transcribed thresholds, which is what a compliance
# question needs. HAS_EXTRACTED_LIMIT would reach the LLM's reading instead.
operating_limit_query = """
WITH node
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)-[:APPLIES_TO]->(a:Aircraft)
OPTIONAL MATCH (a)-[:HAS_SYSTEM]->(sys:System)-[:HAS_SENSOR]->(s:Sensor)-[:HAS_LIMIT]->(ol:OperatingLimit)

WITH node, doc, a,
     COLLECT(DISTINCT {
         sensor: s.type,
         parameter: ol.parameterName,
         max: ol.maxValue,
         unit: ol.unit,
         regime: ol.regime
     })[0..5] AS operating_limits

RETURN
    doc.aircraftType AS aircraft_type,
    operating_limits,
    node.text AS context
"""

# format_operating_limit_record renders each record as clean HTML instead of the
# default str(record), so the results display nicely and read cleanly for the LLM.
limit_retriever = VectorCypherRetriever(
    driver=driver,
    neo4j_database=NEO4J_DATABASE,
    index_name=INDEX_NAME,
    embedder=embedder,
    retrieval_query=operating_limit_query,
    result_formatter=format_operating_limit_record
)

print("VectorCypherRetriever initialized with operating limits query")

In [ ]:
query = "An EGT reading of 700C was recorded during takeoff. Is that within the documented limit for this aircraft?"

rag = GraphRAG(llm=llm, retriever=limit_retriever)
response = rag.search(
    query,
    retriever_config={"top_k": 3},
    return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)

print(f"Query: \"{query}\"\n")
print(f"Number of results: {len(response.retriever_result.items)}\n")
print("=" * 70)
print("\nAnswer:")
print(response.answer)

In [ ]:
# View the graph-connected context with operating limits (rendered as HTML)
from IPython.display import display, HTML

display(HTML("<h4>Context with operating limits:</h4>"))
for item in response.retriever_result.items:
    display(HTML(item.content))

**How this works:**

1. **Semantic Search:** Finds chunks about EGT and temperature limits
2. **Graph Traversal:** Chunk -> Document -> Aircraft -> System -> Sensor -> OperatingLimit
3. **Returns:** Both the relevant text AND the structured operating limit data

**Why this shows the value of graph-connected reference data:** The question asks whether a specific reading (700C) is within limits. A text-only retriever would surface a passage that *mentions* EGT and leave the LLM to infer the threshold from prose, hedging if the exact number was not retrieved. Here the retriever hands the LLM the limit as a structured field (`max`, `unit`), so it can compare 700C against the documented maximum and answer precisely. This is the same reading-versus-limit comparison that powers the agents in Labs 4 and 5, where the live reading comes from Databricks sensor data. Lab 5's `graphrag_node` is built directly on the retrievers you construct here.

> **Note:** These thresholds come from Lab 2's CSV load, so they are present whether or not the extraction run in notebook 01 found anything. That is deliberate: a retriever that answers a compliance question should not depend on what a language model happened to read. What the model did read is reachable in parallel through `HAS_EXTRACTED_LIMIT` to `ExtractedLimit`, and putting the two side by side is how you measure the extraction.

---

# Part 3: Comparing Retrieval Strategies

Let's compare the same query using different retrievers to see the difference in context and answers.

In [ ]:
comparison_query = "Is an EGT reading of 720C within the engine's operating limits?"

print(f"Query: \"{comparison_query}\"")
print("\n" + "=" * 70)

# Basic Vector Retriever
print("\n[1] VECTOR RETRIEVER (text only)")
print("-" * 40)
rag_basic = GraphRAG(llm=llm, retriever=vector_retriever)
response_basic = rag_basic.search(
    comparison_query,
    retriever_config={"top_k": 3},
    return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)
print(response_basic.answer)

# Graph-Enhanced Retriever with adjacent chunks
print("\n" + "=" * 70)
print("\n[2] VECTOR CYPHER RETRIEVER (with adjacent chunks)")
print("-" * 40)
rag_enhanced = GraphRAG(llm=llm, retriever=adjacent_retriever)
response_enhanced = rag_enhanced.search(
    comparison_query,
    retriever_config={"top_k": 3},
    return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)
print(response_enhanced.answer)

# Graph-Enhanced Retriever traversing to Lab 2's canonical OperatingLimit nodes
print("\n" + "=" * 70)
print("\n[3] VECTOR CYPHER RETRIEVER (with operating limits)")
print("-" * 40)
rag_limits = GraphRAG(llm=llm, retriever=limit_retriever)
response_limits = rag_limits.search(
    comparison_query,
    retriever_config={"top_k": 3},
    return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)
print(response_limits.answer)

**Key Differences:**

Compare how each retriever handled "Is an EGT reading of 720C within limits?":

- **[1] VectorRetriever (text only):** Returns chunks that mention EGT. The LLM must infer the threshold from prose and may hedge if the exact number is not in the retrieved text.
- **[2] VectorCypherRetriever (adjacent chunks):** Adds surrounding chunks for more complete procedures, but still relies on the threshold appearing in the text.
- **[3] VectorCypherRetriever (operating limits):** Returns the documented limit as structured data (`max`, `unit`), so the LLM can directly compare 720C against the maximum and answer with confidence.

| Aspect | VectorRetriever | VectorCypherRetriever |
|--------|-----------------|----------------------|
| Context | Raw text chunks | Text + structured graph data |
| Relationships | Implicit in text | Explicit via Cypher traversal |
| Answer completeness | May miss surrounding steps | Includes adjacent procedures |
| Numeric thresholds | Inferred from prose, may be vague | Returned as structured fields |
| Best for | Quick lookups | Complete procedures, compliance checks |

**When to use each:**
- **VectorRetriever**: Simple semantic search, quick answers, fact lookups
- **VectorCypherRetriever**: Procedural questions, troubleshooting flows, compliance checks against documented limits, when you need context from related chunks or graph entities

## Try Your Own Queries

Experiment with different maintenance questions. Here are some to try:

In [ ]:
# Try different maintenance queries
sample_queries = [
    "What are the vibration limits that require engine shutdown?",
    "How do I check for hydraulic fluid contamination?",
    "What oil analysis levels indicate bearing wear?",
    "When should I perform a borescope inspection?"
]

# Use the adjacent retriever for better context
rag = GraphRAG(llm=llm, retriever=adjacent_retriever)

for query in sample_queries:
    print(f"\nQ: {query}")
    print("-" * 70)
    response = rag.search(
        query,
        retriever_config={"top_k": 2},
        return_context=True,
        response_fallback="No relevant maintenance procedures found.",
    )
    # Print first 500 chars of answer
    answer = response.answer[:500] + "..." if len(response.answer) > 500 else response.answer
    print(f"A: {answer}\n")

## Summary

In this notebook, you learned retrieval strategies for aircraft maintenance GraphRAG:

**Part 1 - Vector Retriever:**
1. Simple semantic search using vector embeddings
2. GraphRAG pipeline combining retrieval with LLM generation
3. Diagnostic inspection of search results

**Part 2 - Vector Cypher Retriever:**
4. Document metadata enrichment
5. Adjacent chunk retrieval for complete procedures
6. Connecting to aircraft topology via APPLIES_TO traversal (systems, components)
7. Reaching Lab 2's canonical OperatingLimit nodes through graph traversal

**Part 3 - Comparison:**
8. Understanding when to use each approach
9. Trade-offs between simplicity and context richness

| Retriever | Context | Graph Traversal | Best For |
|-----------|---------|-----------------|----------|
| VectorRetriever | Raw text chunks | None | Quick lookups, fact retrieval |
| VectorCypherRetriever (adjacent) | Text + surrounding chunks | NEXT_CHUNK | Procedural questions, troubleshooting |
| VectorCypherRetriever (topology) | Text + systems/components | APPLIES_TO, HAS_SYSTEM | Aircraft-specific questions |
| VectorCypherRetriever (limits) | Text + operating limits | Full chain to OperatingLimit | Threshold/specification queries |

---

**Your knowledge graph now combines:**
- **Structured topology**: Aircraft -> System -> Component hierarchy
- **Semantic search**: Maintenance manual chunks with embeddings
- **Reference data**: Lab 2's canonical `OperatingLimit` nodes, linked to the sensors they judge
- **Extracted entities**: `ExtractedLimit` nodes read out of the manual text, linked to the same sensors in parallel
- **GraphRAG retrieval**: Context-aware answers grounded in documentation and structured data

In [ ]:
# Cleanup
neo4j.close()